In [ ]:
import os
import pandas as pd
from tqdm import tqdm

# 确保存在用于保存拆分后的文件的文件夹
output_folders = ['discharge', 'charge']
for folder in output_folders:
    if not os.path.exists(folder):
        os.makedirs(folder)

# 文件名列表
file_names = [f'data{i}.xlsx' for i in range(1, 16)]
print(file_names)

# 使用 tqdm 显示进度条
for file_name in tqdm(file_names, desc="处理文件"):
    # 检查文件是否存在
    if not os.path.exists(file_name):
        print(f"文件 {file_name} 不存在，跳过该文件。")
        continue  # 跳过不存在的文件

    # 读取 Excel 文件
    df = pd.read_excel(file_name)

    # 确保第三列存在
    if df.shape[1] < 3:
        print(f"{file_name} 没有第三列，跳过该文件。")
        continue

    # 获取第三列数据
    col_3 = df.iloc[:, 2]
    
    # 初始化拆分变量
    current_state = None
    split_data = []
    temp_df = pd.DataFrame()
    segment_counter = {'discharge': 1, 'charge': 1}

    # 遍历第三列数据进行拆分
    for index, value in tqdm(col_3.items(), desc=f"处理 {file_name}", leave=False):
        if value != current_state:
            if not temp_df.empty:
                split_data.append((current_state, temp_df))
                temp_df = pd.DataFrame()
            current_state = value
        temp_df = pd.concat([temp_df, df.iloc[[index]]], ignore_index=True)
    
    # 添加最后一个分段
    if not temp_df.empty:
        split_data.append((current_state, temp_df))

    # 保存拆分后的数据
    for state, data in split_data:
        if state == 3:
            segment_type = 'discharge'
            output_folder = 'discharge'
        elif state == 1:
            segment_type = 'charge'
            output_folder = 'charge'
        else:
            continue  # 跳过不符合条件的状态

        # 创建文件名，附加片段类型和序号
        output_file = os.path.join(
            output_folder,
            f'{os.path.splitext(file_name)[0]}_{segment_type}{segment_counter[segment_type]}.xlsx'
        )
        data.to_excel(output_file, index=False)
        segment_counter[segment_type] += 1

    print(f"{file_name} 已处理完毕。")

print("所有文件已拆分完毕。")
